# 예제 franka_ex05: FR3 Cartesian 경로 — 직선 보간 (self-contained)

Franka FR3(7-DOF) 의 끝단(`fr3_hand_tcp`) 을 **공간상의 직선** 으로 이동시키는 예제.
6-DOF 용 `ex05_cartesian_path.py` 를 FR3 워크스페이스에 맞춰 옮겨온 단일 노트북 버전이다.

**6-DOF 예제와 다른 점**
- 끝단 링크: `fr3_hand_tcp` — TCP(손가락 사이 중심)
- planning group: `fr3_arm`
- 워크스페이스: FR3 reach ~855mm 이므로 정사각형 한 변을 15cm 로 키우고, base 로부터 ~40~55cm 사이에 둔다
- 7-DOF redundancy 덕에 같은 직선 경로도 IK 해가 여러 개 → 특이점 (singularity) 회피가 6-DOF 보다 잘 된다
- `home`(올-제로) 자세가 SRDF 에 없다 → 시작/복귀는 `ready`
- Gazebo Sim 환경이므로 `use_sim_time=True`

**학습 내용**
- `GetCartesianPath` 서비스 직접 호출 — `MoveGroup` 액션과 어떻게 다른지
- `max_step` 파라미터로 직선 보간 간격 조절
- `fraction` (달성률) 으로 경로 일부만 풀린 경우 감지
- `ExecuteTrajectory` 액션으로 계획된 궤적만 실행 (계획 ≠ 실행 분리)
- RViz2 `MarkerArray` 로 정사각형/직선 경로 **미리보기**

**워크플로**

각 경로마다 세 단계로 나뉜다:

1. **미리보기 단계** — 원하는 waypoints 를 RViz2 에 LINE_STRIP / SPHERE 로 표시. 로봇은 가만히.
2. **계획 단계** — `compute_cartesian_path(waypoints, max_step)` 호출. 서비스가 IK 를 풀어 trajectory 와 fraction 을 반환.
3. **실행 단계** — fraction 이 충분히 높으면 `execute_trajectory(traj)` 로 실제 이동. 실행 결과(성공/실패)에 따라 마커 색상이 바뀐다.


## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch (`demo.launch.xml` 등) 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/cartesian_path_markers` 로 설정**한다.
이 토픽으로 정사각형/직선 경로의 미리보기 마커가 발행된다.
Fixed Frame 은 `fr3_link0` 로 둔다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab franka_ex05_cartesian_path.ipynb
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).


## 1. 로봇 상수 정의

이 값들은 `franka_description/robots/fr3/fr3.srdf.xacro` 에서 생성되는 SRDF 와 일치한다.

In [1]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC      = '/cartesian_path_markers'

## 2. ROS 2 초기화와 노드 생성

이 예제는 `MoveGroup` 액션 (조인트/자세 목표용), `ExecuteTrajectory` 액션 (계획만 된 궤적 실행용),
`compute_cartesian_path` 서비스 (직선 보간), 그리고 마커 퍼블리셔 / `joint_states` 구독자를 같은 노드에 모두 붙인다.
Gazebo Sim 과 시간을 맞추기 위해 `use_sim_time=True` 를 준다.

### 2-1. import (이 섹션에서 처음 쓰이는 것들)

In [2]:
import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from moveit_msgs.srv import GetCartesianPath
from visualization_msgs.msg import MarkerArray

### 2-2. `rclpy` 초기화

한 프로세스에서 한 번만 init 가능하므로, 두 번째 실행은 무시되도록 `try/except` 로 감싼다.

In [3]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우 무시

### 2-3. 노드, 액션/서비스 클라이언트, 구독자/퍼블리셔

| 핸들 | 역할 |
|---|---|
| `move_client` | `MoveGroup` 액션 — 조인트/자세 목표로 이동 (계획+실행 한 번에) |
| `execute_client` | `ExecuteTrajectory` 액션 — 이미 계획된 궤적만 실행 |
| `cart_client` | `compute_cartesian_path` 서비스 — waypoints 로 직선 경로 계획 |
| `marker_pub` | `/cartesian_path_markers` 토픽 퍼블리셔 — RViz 시각화 |

In [4]:
node = Node(
    'franka_ex05_cartesian_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client    = ActionClient(node, MoveGroup, 'move_action')
execute_client = ActionClient(node, ExecuteTrajectory, 'execute_trajectory')
cart_client    = node.create_client(GetCartesianPath, 'compute_cartesian_path')
marker_pub     = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex05 노트북 노드 생성 완료 ===')

[INFO] [1778234702.228520975] [franka_ex05_cartesian_demo]: === franka_ex05 노트북 노드 생성 완료 ===


True

## 3. 액션 서버 / 서비스 / `/joint_states` 준비 대기

세 개의 액션·서비스 핸들과 첫 `joint_states` 메시지가 모두 준비되어야 안전하게 진행할 수 있다.
이 셀에서 `time` 모듈이 처음 쓰이므로 함께 import 한다.

In [6]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    if not execute_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('ExecuteTrajectory 액션 서버 연결 실패')
    if not cart_client.wait_for_service(timeout_sec=timeout_sec):
        raise RuntimeError('compute_cartesian_path 서비스 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('move/execute action + cartesian svc + /joint_states 준비됨')

wait_for_ready()

[INFO] [1778234735.973301953] [franka_ex05_cartesian_demo]: move/execute action + cartesian svc + /joint_states 준비됨


## 4. SRDF 에서 `ready` 자세 읽어오기

`move_group` 의 `robot_description_semantic` 파라미터에서 SRDF XML 을 받아
`<group_state name="ready" group="fr3_arm">` 의 조인트 값들을 dict 로 추출한다.
여기서 `AsyncParameterClient` 와 `xml.etree.ElementTree` 가 처음 쓰이므로 같이 import 한다.
FR3 SRDF 에는 `home` 이 없고 `ready` / `extended` 만 있다.

In [7]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

[INFO] [1778234737.126193281] [franka_ex05_cartesian_demo]: ready: {'fr3_joint1': 0.0, 'fr3_joint2': -0.7853981633974483, 'fr3_joint3': 0.0, 'fr3_joint4': -2.356194490192345, 'fr3_joint5': 0.0, 'fr3_joint6': 1.5707963267948966, 'fr3_joint7': 0.7853981633974483}


True

## 5. Pose 헬퍼 — Euler ↔ Quaternion

`Pose` 메시지의 방향은 쿼터니언이다. 사람이 쓰기 편한 `roll/pitch/yaw` (rad) 을 변환하는 헬퍼.
이 셀에서 `math`, `tf_transformations`, `Pose` / `Point` / `Quaternion` 이 처음 쓰이므로 함께 import 한다.

In [8]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Point, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

## 6. `MoveGroup` 액션 헬퍼 — 조인트 목표 / 자세 목표

Cartesian 경로 계획 전에 `ready` (조인트 목표) 와 정사각형 시작점 (자세 목표) 으로 옮겨야 한다.
franka_ex03/04 와 동일한 패턴으로 작은 헬퍼 함수들로 분리한다.

여기서 `Constraints`, `JointConstraint`, `PositionConstraint`, `OrientationConstraint`,
`BoundingVolume`, `SolidPrimitive`, `MotionPlanRequest`, `PlanningOptions`,
`MoveItErrorCodes`, `Vector3` 가 처음 쓰이므로 함께 import 한다.

In [9]:
from moveit_msgs.msg import (
    Constraints, JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
    MotionPlanRequest, PlanningOptions, MoveItErrorCodes,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose: Pose, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    oc.orientation = pose.orientation
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float, acc: float,
                       attempts: int = 5, plan_time: float = 10.0) -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    return req

def send_move_goal_and_wait(req: MotionPlanRequest) -> int:
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=False, replan=True, replan_attempts=3)
    send_future = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, send_future)
    handle = send_future.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED
    result_future = handle.get_result_async()
    rclpy.spin_until_future_complete(node, result_future)
    return result_future.result().result.error_code.val

def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val = send_move_goal_and_wait(req)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'joint goal 실패 error_code={code_val}')
    return ok

def go_to_pose_goal(pose: Pose, vel: float = 0.2, acc: float = 0.2) -> bool:
    req = make_plan_request(vel, acc)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val = send_move_goal_and_wait(req)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'pose goal 실패 error_code={code_val} (IK 해 없음 가능)')
    return ok

## 7. Cartesian 경로 계획 — `compute_cartesian_path` 서비스 직접 호출

이 예제의 핵심.

`MoveGroup` 액션은 *시작* 과 *목표* 만 주면 알아서 경로를 찾는 PRM/RRT 류 플래너 — 결과 trajectory 가 직선이 아닐 수 있다.
반면 `compute_cartesian_path` 서비스는 **공간상의 직선** 으로 보간한 trajectory 를 반환한다.

| 입력 | 의미 |
|---|---|
| `waypoints` | 끝단이 거쳐야 할 `Pose` 들 — 사이는 직선으로 채운다 |
| `max_step` | 직선 보간 간격 (m) — 작을수록 부드럽고 IK 호출 횟수↑ |
| `avoid_collisions` | True 면 IK 단계에서 충돌 검사 |
| `start_state` | 보통 현재 `joint_states` (없으면 SRDF 기본값) |

응답은 `(trajectory, fraction)`. `fraction == 1.0` 이면 모든 waypoint 까지 IK 풀었다는 뜻.
중간에 특이점/IK 실패 가 있으면 0.x 가 나오므로 임계값(예: 0.8) 으로 필터링한다.

`RobotState` 가 처음 쓰이므로 같이 import 한다.

In [10]:
from moveit_msgs.msg import RobotState

def get_current_robot_state() -> RobotState:
    rs = RobotState()
    if joint_state['msg'] is not None:
        rs.joint_state = joint_state['msg']
    return rs

def compute_cartesian_path(waypoints, max_step: float = 0.01,
                            avoid_collisions: bool = True,
                            vel: float = 0.2, acc: float = 0.2):
    '''waypoints 사이를 직선 보간하여 (trajectory, fraction) 반환.'''
    request = GetCartesianPath.Request()
    request.header.frame_id = REFERENCE_FRAME
    request.group_name = PLANNING_GROUP
    request.link_name = END_EFFECTOR_LINK
    request.waypoints = list(waypoints)
    request.max_step = max_step
    request.avoid_collisions = avoid_collisions
    request.max_velocity_scaling_factor = vel
    request.max_acceleration_scaling_factor = acc
    request.start_state = get_current_robot_state()

    future = cart_client.call_async(request)
    rclpy.spin_until_future_complete(node, future)
    response = future.result()

    if response.error_code.val == MoveItErrorCodes.SUCCESS:
        node.get_logger().info(
            f'Cartesian 경로 계획 성공 (달성률: {response.fraction*100:.1f}%)'
        )
        return response.solution, response.fraction
    node.get_logger().error(
        f'Cartesian 계획 실패 error_code={response.error_code.val}'
    )
    return None, 0.0

## 8. 계획된 궤적 실행 — `ExecuteTrajectory` 액션

`MoveGroup` 액션과 달리 Cartesian 서비스는 **계획만** 한다. 결과 trajectory 를 실제로 컨트롤러에 보내려면 `ExecuteTrajectory` 액션이 필요하다.

In [11]:
def execute_trajectory(trajectory) -> bool:
    goal = ExecuteTrajectory.Goal()
    goal.trajectory = trajectory

    send_future = execute_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, send_future)
    handle = send_future.result()
    if handle is None or not handle.accepted:
        node.get_logger().error('ExecuteTrajectory 목표 거부됨')
        return False

    result_future = handle.get_result_async()
    rclpy.spin_until_future_complete(node, result_future)
    code_val = result_future.result().result.error_code.val
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'궤적 실행 실패 error_code={code_val}')
    return ok

## 9. RViz 마커 헬퍼 — 경로 미리보기 + 결과 표시

waypoints 를 `LINE_STRIP` 으로 잇고, 각 waypoint 는 `SPHERE_LIST` 로 점을 찍는다.
시작점은 큰 노란 구체로 강조하고, 각 점에는 라벨 텍스트를 붙인다.

| 색상 | 의미 |
|---|---|
| 시안 (`COLOR_SQUARE`) | 정사각형 미리보기 |
| 주황 (`COLOR_DESCENT`) | 직선 하강 미리보기 |
| 노랑 (`COLOR_START`) | 경로 시작점 |
| 초록 (`COLOR_SUCCESS`) | 실행 성공 후 |
| 빨강 (`COLOR_FAIL`) | 실행 실패 후 |
| 흰색 (`COLOR_TEXT`) | 라벨 텍스트 |

`publish_path_markers(...)` 한 번 호출 = 새 경로의 미리보기 마커 묶음을 발행.
`publish_result_marker(...)` 한 번 호출 = 같은 ns 의 LINE_STRIP 색상을 결과에 맞게 바꾸고 결과 텍스트 추가.

`Marker`, `ColorRGBA` 가 처음 쓰이므로 같이 import 한다.

In [12]:
from visualization_msgs.msg import Marker
from std_msgs.msg import ColorRGBA

COLOR_SQUARE  = ColorRGBA(r=0.2, g=0.8, b=1.0, a=0.9)   # 시안
COLOR_DESCENT = ColorRGBA(r=1.0, g=0.6, b=0.0, a=0.9)   # 주황
COLOR_START   = ColorRGBA(r=1.0, g=1.0, b=0.0, a=0.9)   # 노랑
COLOR_SUCCESS = ColorRGBA(r=0.0, g=1.0, b=0.0, a=0.9)   # 초록
COLOR_FAIL    = ColorRGBA(r=1.0, g=0.0, b=0.0, a=0.9)   # 빨강
COLOR_TEXT    = ColorRGBA(r=1.0, g=1.0, b=1.0, a=1.0)   # 흰

# 누적 마커 상태 (덮어쓰기 위해 보관)
_markers = MarkerArray()
_marker_id = {'next': 0}

def _next_id() -> int:
    _marker_id['next'] += 1
    return _marker_id['next']

def publish_path_markers(waypoints, start_pose: Pose,
                          ns: str, color: ColorRGBA, label: str) -> None:
    '''waypoints 를 LINE_STRIP + SPHERE_LIST + 시작점 SPHERE + 라벨로 발행.'''
    stamp = node.get_clock().now().to_msg()
    pts = [start_pose.position] + [wp.position for wp in waypoints]

    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = ns + '_path'
    line.id = _next_id()
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.pose.orientation.w = 1.0
    line.scale.x = 0.008
    line.color = color
    line.points = [Point(x=p.x, y=p.y, z=p.z) for p in pts]
    _markers.markers.append(line)

    spheres = Marker()
    spheres.header.frame_id = REFERENCE_FRAME
    spheres.header.stamp = stamp
    spheres.ns = ns + '_waypoints'
    spheres.id = _next_id()
    spheres.type = Marker.SPHERE_LIST
    spheres.action = Marker.ADD
    spheres.pose.orientation.w = 1.0
    spheres.scale = Vector3(x=0.03, y=0.03, z=0.03)
    spheres.color = color
    spheres.points = [Point(x=p.x, y=p.y, z=p.z) for p in pts]
    _markers.markers.append(spheres)

    start_sphere = Marker()
    start_sphere.header.frame_id = REFERENCE_FRAME
    start_sphere.header.stamp = stamp
    start_sphere.ns = ns + '_start'
    start_sphere.id = _next_id()
    start_sphere.type = Marker.SPHERE
    start_sphere.action = Marker.ADD
    start_sphere.pose.position = Point(x=start_pose.position.x,
                                       y=start_pose.position.y,
                                       z=start_pose.position.z)
    start_sphere.pose.orientation.w = 1.0
    start_sphere.scale = Vector3(x=0.05, y=0.05, z=0.05)
    start_sphere.color = COLOR_START
    _markers.markers.append(start_sphere)

    for i, p in enumerate(pts):
        text = Marker()
        text.header.frame_id = REFERENCE_FRAME
        text.header.stamp = stamp
        text.ns = ns + '_text'
        text.id = _next_id()
        text.type = Marker.TEXT_VIEW_FACING
        text.action = Marker.ADD
        text.pose.position = Point(x=p.x, y=p.y, z=p.z + 0.07)
        text.pose.orientation.w = 1.0
        text.scale.z = 0.05
        text.color = COLOR_TEXT
        text.text = f'{label} Start' if i == 0 else f'P{i}'
        _markers.markers.append(text)

    marker_pub.publish(_markers)

def publish_result_marker(ns: str, success: bool, label: str) -> None:
    '''같은 ns 의 LINE_STRIP 색을 결과 색으로 바꾸고 결과 텍스트 추가.'''
    stamp = node.get_clock().now().to_msg()
    color = COLOR_SUCCESS if success else COLOR_FAIL
    center = None
    for m in _markers.markers:
        if m.ns == ns + '_path':
            m.color = color
            m.header.stamp = stamp
            if m.points:
                mid = m.points[len(m.points) // 2]
                center = Point(x=mid.x, y=mid.y, z=mid.z + 0.08)
    if center is not None:
        result = Marker()
        result.header.frame_id = REFERENCE_FRAME
        result.header.stamp = stamp
        result.ns = ns + '_result'
        result.id = _next_id()
        result.type = Marker.TEXT_VIEW_FACING
        result.action = Marker.ADD
        result.pose.position = center
        result.pose.orientation.w = 1.0
        result.scale.z = 0.06
        result.color = color
        result.text = f'{label}: {"OK" if success else "FAIL"}'
        _markers.markers.append(result)
    marker_pub.publish(_markers)

## 10. 시나리오 시작 — `ready` 자세로 초기화

Cartesian 경로 계획 전에 잘 보이는 자세로 옮긴다.

In [13]:
node.get_logger().info('--- ready 자세로 초기 이동 ---')
go_to_joint_goal(ready_target)
time.sleep(1.0)

[INFO] [1778234746.452586223] [franka_ex05_cartesian_demo]: --- ready 자세로 초기 이동 ---


## 11. 정사각형 시작점으로 이동 (Pose Goal)

정사각형은 끝단이 **아래를 향한 자세** (`roll=π`) 로 X-Y 평면 (`z=0.40`) 에 그린다.
시작점은 (`x=0.40, y=-0.075, z=0.40`) — base 로부터 ~40cm 전방, 7cm 우측.

7-DOF redundancy 덕에 Cartesian 경로가 잘 풀리는 자세이기도 하다 (조인트가 한계 근처가 아님).

In [14]:
SQUARE_Z = 0.40            # 정사각형 평면 z
SIDE     = 0.15            # 한 변 15cm
SX, SY   = 0.40, -SIDE / 2  # 시작점 (X-Y)

square_orientation = euler_to_quaternion(math.pi, 0.0, 0.0)  # 그리퍼 아래

start_pose = Pose()
start_pose.position = Point(x=SX, y=SY, z=SQUARE_Z)
start_pose.orientation = square_orientation

node.get_logger().info(
    f'--- 정사각형 시작점으로 이동: ({SX:.2f}, {SY:.3f}, {SQUARE_Z:.2f}) ---'
)
ok = go_to_pose_goal(start_pose)
if not ok:
    node.get_logger().error('시작점 이동 실패 — ready 로 복귀')
    go_to_joint_goal(ready_target)
time.sleep(1.0)

[INFO] [1778234749.893915973] [franka_ex05_cartesian_demo]: --- 정사각형 시작점으로 이동: (0.40, -0.075, 0.40) ---


## 12. 정사각형 Cartesian 경로 — 계획 + 미리보기

X-Y 평면에서 한 변 15cm 정사각형을 시계 방향으로 그린다 (시작점 → +y → +x → -y → -x → 시작점 복귀).
각 변 사이의 직선은 `compute_cartesian_path` 가 `max_step=0.02` 단위로 IK 를 풀어 채운다.

In [17]:
waypoints = [
    # +Y 방향
    Pose(position=Point(x=SX,        y=SY + SIDE, z=SQUARE_Z),
         orientation=square_orientation),
    # +X 방향
    Pose(position=Point(x=SX + SIDE, y=SY + SIDE, z=SQUARE_Z),
         orientation=square_orientation),
    # -Y 방향
    Pose(position=Point(x=SX + SIDE, y=SY,        z=SQUARE_Z),
         orientation=square_orientation),
    # -X 방향 (시작점 복귀)
    Pose(position=Point(x=SX,        y=SY,        z=SQUARE_Z),
         orientation=square_orientation),
]

publish_path_markers(waypoints, start_pose,
                     ns='square', color=COLOR_SQUARE, label='Square')

trajectory, fraction = compute_cartesian_path(waypoints, max_step=0.02)
node.get_logger().info(f'정사각형 달성률: {fraction*100:.1f}%')

[INFO] [1778234803.694490473] [franka_ex05_cartesian_demo]: Cartesian 경로 계획 성공 (달성률: 100.0%)
[INFO] [1778234803.695626916] [franka_ex05_cartesian_demo]: 정사각형 달성률: 100.0%


True

### 12-2. 계획된 궤적 실행

`fraction` 이 0.8 이상이면 실행. 그렇지 않으면 경로 일부만 풀린 상태이므로 안전하게 스킵.

In [18]:
if trajectory is not None and fraction > 0.8:
    node.get_logger().info('--- 정사각형 경로 실행 ---')
    ok = execute_trajectory(trajectory)
    publish_result_marker('square', ok, 'Square')
else:
    node.get_logger().warn(
        f'달성률이 낮아 정사각형 실행을 생략 ({fraction*100:.1f}%)'
    )
    publish_result_marker('square', False, 'Square')
time.sleep(1.0)

[INFO] [1778234812.833144208] [franka_ex05_cartesian_demo]: --- 정사각형 경로 실행 ---


## 13. 직선 하강 시연

정사각형 시작점에서 **직선으로 15cm 아래** 로 내려보낸다 (`z` 값만 단조감소).
2cm 간격으로 waypoint 를 7 개 두면 보간이 매끄럽다.

In [19]:
DESCENT = 0.15
descent_start = Pose()
descent_start.position = Point(x=SX, y=SY, z=SQUARE_Z)
descent_start.orientation = square_orientation

descent_waypoints = []
for i in range(1, 8):
    p = Pose()
    p.position = Point(x=SX, y=SY, z=SQUARE_Z - i * (DESCENT / 7))
    p.orientation = square_orientation
    descent_waypoints.append(p)

publish_path_markers(descent_waypoints, descent_start,
                     ns='descent', color=COLOR_DESCENT, label='Descent')

traj2, frac2 = compute_cartesian_path(descent_waypoints, max_step=0.02)
node.get_logger().info(f'직선 하강 달성률: {frac2*100:.1f}%')

if traj2 is not None and frac2 > 0.8:
    ok2 = execute_trajectory(traj2)
    publish_result_marker('descent', ok2, 'Descent')
else:
    publish_result_marker('descent', False, 'Descent')
time.sleep(1.0)

[INFO] [1778234830.473213225] [franka_ex05_cartesian_demo]: Cartesian 경로 계획 성공 (달성률: 100.0%)
[INFO] [1778234830.474568371] [franka_ex05_cartesian_demo]: 직선 하강 달성률: 100.0%


## 14. `ready` 자세로 복귀

In [20]:
node.get_logger().info('--- ready 자세로 복귀 ---')
go_to_joint_goal(ready_target)
node.get_logger().info('=== franka_ex05 완료! ===')

[INFO] [1778234834.739470440] [franka_ex05_cartesian_demo]: --- ready 자세로 복귀 ---
[INFO] [1778234836.619663233] [franka_ex05_cartesian_demo]: === franka_ex05 완료! ===


True

## 15. 정리

노트북을 닫기 전에 노드와 rclpy 를 안전하게 정리한다.

In [21]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass